# Whisper Fine-tuning
This notebook demonstrates how to fine-tune OpenAI's Whisper model for multilingual automatic speech recognition (ASR) using Hugging Face Transformers and Datasets. It covers environment setup, configuration, training, evaluation, and launching a Gradio demo.

In [ ]:
# Install required packages (if running in a fresh environment)
! pip install gradio evaluate jiwer

In [ ]:
# Import required libraries
import os
import sys
import torch
import yaml
import random
from datetime import datetime
import json
import gradio as gr
from datasets import load_dataset, DatasetDict, Audio
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor,
    WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
)
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
torch.cuda.is_available()

In [ ]:
from huggingface_hub import login
hf_token = "token_here"
login(token=hf_token)

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        if "attention_mask" not in batch:
            batch["attention_mask"] = torch.ones(batch["input_features"].shape[:-1], dtype=torch.long)
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

## Configuration
Set your training and model parameters here. You can edit these values as needed for your experiment.

In [ ]:
# Example configuration (edit as needed)
lang = 'as'  # Language code (e.g., 'as' for Assamese)
dataset_name = 'google/fleurs'  # or 'google/fleurs'
model_name = 'whisper-small'
dataset_cache = '/kaggle/datasets'
model_cache = '/kaggle/models'
whisper_pretrained = f"openai/{model_name}"
checkpoint_name = f"{dataset_name}-{model_name}-{lang}"
checkpoint_dir = f"/kaggle/working/checkpoints/{checkpoint_name}"
max_steps = 4000
gpu_device = None  # Set to a string like '0' to select a specific GPU, or None for default

In [ ]:
# Set CUDA device if needed
if gpu_device is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_device)
    print(f"Set CUDA_VISIBLE_DEVICES to {gpu_device}")
    torch.cuda.set_device(0)

In [ ]:
# Feature extractor, tokenizer, processor
feature_extractor = WhisperFeatureExtractor.from_pretrained(whisper_pretrained, cache_dir=model_cache)
tokenizer = WhisperTokenizer.from_pretrained(whisper_pretrained, language=lang, task="transcribe", cache_dir=model_cache)
processor = WhisperProcessor.from_pretrained(whisper_pretrained, language=lang, task="transcribe", cache_dir=model_cache)

In [ ]:
# Dataset loading and preparation
indic_langs = {
    'as', 'bn', 'gu', 'hi', 'kn', 'ks', 'ml', 'mr', 'ne', 'or', 'pa', 'sa', 'sd', 'ta', 'te', 'ur',
    'mai', 'gom', 'doi', 'bho', 'brx', 'sat', 'mni', 'kok', 'lus', 'kha', 'new', 'raj', 'mag', 'hne', 'lep', 'bpy', 'wbq', 'unr', 'sck', 'gbm', 'awa', 'bhb', 'bhi', 'bjj', 'bns', 'bpy', 'bto', 'ccp', 'chh', 'hne', 'hoc', 'khn', 'kru', 'mwr', 'noe', 'ory', 'pan', 'pnb', 'raj', 'rjs', 'sck', 'snd', 'unr', 'wbr'
}
is_fleurs = 'fleurs' in dataset_name.lower()
fleurs_lang = f"{lang}_in" if is_fleurs and lang in indic_langs and not lang.endswith('_in') else lang
if is_fleurs:
    dataset = DatasetDict()
    dataset["train"] = load_dataset(dataset_name, fleurs_lang, split="train+validation", cache_dir=dataset_cache, trust_remote_code=True)
    dataset["test"] = load_dataset(dataset_name, fleurs_lang, split="test", cache_dir=dataset_cache, trust_remote_code=True)
    def prepare_fleurs(batch):
        audio = batch["audio"]
        batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
        batch["labels"] = tokenizer(batch["transcription"]).input_ids
        return batch
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
    dataset = dataset.map(prepare_fleurs, remove_columns=dataset["train"].column_names, num_proc=1)
else:
    dataset = DatasetDict()
    dataset["train"] = load_dataset(dataset_name, lang, split="train+validation", cache_dir=dataset_cache, trust_remote_code=True)
    dataset["test"] = load_dataset(dataset_name, lang, split="test", cache_dir=dataset_cache, trust_remote_code=True)
    dataset = dataset.remove_columns([
        "accent", "age", "client_id", "down_votes", "gender", "locale", "path", "segment", "up_votes"
    ])
    def prepare_common_voice(batch):
        audio = batch["audio"]
        batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
        batch["labels"] = tokenizer(batch["sentence"]).input_ids
        return batch
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
    dataset = dataset.map(prepare_common_voice, remove_columns=dataset["train"].column_names, num_proc=1)

In [ ]:
# Filter out samples with too-long labels (max 448 tokens)
max_label_length = 448
def filter_long_labels(batch):
    return len(batch["labels"]) <= max_label_length

dataset["train"] = dataset["train"].filter(filter_long_labels)
dataset["test"] = dataset["test"].filter(filter_long_labels)
print("Filtered out samples with labels exceeding 448 tokens.")

In [ ]:
# Load model
model = WhisperForConditionalGeneration.from_pretrained(whisper_pretrained, cache_dir=model_cache)
# Move generation parameters from model.config to model.generation_config
for k in [
    'max_length', 'min_length', 'do_sample', 'early_stopping', 'num_beams', 'temperature',
    'top_k', 'top_p', 'repetition_penalty', 'length_penalty', 'no_repeat_ngram_size',
    'encoder_no_repeat_ngram_size', 'bad_words_ids', 'num_return_sequences',
    'forced_bos_token_id', 'forced_eos_token_id', 'remove_invalid_values',
    'exponential_decay_length_penalty', 'suppress_tokens', 'begin_suppress_tokens']:
    if hasattr(model.config, k):
        setattr(model.generation_config, k, getattr(model.config, k))
model.generation_config.language = lang
model.generation_config.task = "transcribe"
model.config.use_cache = False  # For gradient checkpointing compatibility

In [ ]:
# torch.cuda.set_device(1)
# torch.cuda.empty_cache()
# torch.cuda.set_device(0)
# torch.cuda.empty_cache()
# !nvidia-smi

In [ ]:
# Data collator and metrics
collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, decoder_start_token_id=model.config.decoder_start_token_id)
metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=checkpoint_dir,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=max_steps,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy='steps',
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True
)

In [ ]:
# Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

In [ ]:
# Train the model (resume if checkpoint exists)
processor.save_pretrained(training_args.output_dir) # Save processor configuration

checkpoint_found = False
if os.path.isdir(checkpoint_dir):
    for entry in os.listdir(checkpoint_dir):
        if entry.startswith('checkpoint-') and os.path.isdir(os.path.join(checkpoint_dir, entry)):
            checkpoint_found = True
            break
if checkpoint_found:
    print(f"Resuming training from last checkpoint in {checkpoint_dir}...")
    trainer.train(resume_from_checkpoint=True)
else:
    print(f"No valid checkpoint found in {checkpoint_dir}. Starting training from scratch...")
    trainer.train()

In [ ]:
# Evaluate on test set and print final WER
eval_results = trainer.evaluate()
print("Training complete. Best model saved at:", training_args.output_dir)
print(f"Final WER on test set: {eval_results.get('eval_wer', 'N/A')}")

In [ ]:
# Push to Hugging Face Hub
push_kwargs = {
    "dataset_tags": f"{dataset_name}",
    "dataset": "Common Voice 11.0",  # or your dataset name
    "dataset_args": f"config: {lang}, split: test",
    "language": f"{lang}",
    "model_name": f"{checkpoint_name} - Fine-tuned",
    "finetuned_from": whisper_pretrained,
    "tasks": "automatic-speech-recognition",
}
print("Pushing model and processor to the Hugging Face Hub...")
trainer.push_to_hub(**push_kwargs)
print("Push to hub complete.")

## Evaluation and Gradio Demo
You can use the following code to evaluate a checkpoint or launch a Gradio ASR demo interactively.

In [ ]:
def evaluate_checkpoint(
    checkpoint_path,
    dataset_name,
    lang,
    dataset_cache,
    model_cache=None,
    is_pretrained=False
):
    from transformers import WhisperForConditionalGeneration, WhisperProcessor, WhisperFeatureExtractor, WhisperTokenizer
    import torch
    from datasets import load_dataset, Audio
    import evaluate
    if is_pretrained:
        processor = WhisperProcessor.from_pretrained(checkpoint_path, language=lang, task="transcribe")
        feature_extractor = WhisperFeatureExtractor.from_pretrained(checkpoint_path)
        tokenizer = WhisperTokenizer.from_pretrained(checkpoint_path, language=lang, task="transcribe")
        model = WhisperForConditionalGeneration.from_pretrained(checkpoint_path)
    else:
        processor = WhisperProcessor.from_pretrained(checkpoint_path)
        feature_extractor = WhisperFeatureExtractor.from_pretrained(checkpoint_path)
        tokenizer = WhisperTokenizer.from_pretrained(checkpoint_path)
        model = WhisperForConditionalGeneration.from_pretrained(checkpoint_path)
    model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    test_set = load_dataset(dataset_name, lang, split="test", cache_dir=dataset_cache, trust_remote_code=True)
    test_set = test_set.cast_column("audio", Audio(sampling_rate=16000))
    samples = list(test_set)
    inputs = [feature_extractor(s["audio"]["array"], sampling_rate=16000).input_features[0] for s in samples]
    input_features = torch.tensor(inputs).unsqueeze(1) if len(inputs[0].shape) == 1 else torch.tensor(inputs)
    input_features = input_features.to(device)
    with torch.no_grad():
        predicted_ids = model.generate(input_features)
    pred_str = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)
    label_str = [s["sentence"] for s in samples]
    metric = evaluate.load("wer")
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    print(f"\nWER: {wer:.2f}%\n")

In [ ]:
def gradio_transcribe_interface(
    checkpoint_path,
    lang,
    model_cache=None,
    is_pretrained=False
):
    from transformers import WhisperForConditionalGeneration, WhisperProcessor
    import torch
    import evaluate
    import numpy as np
    processor = WhisperProcessor.from_pretrained(checkpoint_path)
    model = WhisperForConditionalGeneration.from_pretrained(checkpoint_path)
    model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    metric = evaluate.load("wer")
    def transcribe_and_score(audio, reference_text):
        if audio is None:
            return "No audio provided.", "-"
        if isinstance(audio, tuple):
            sr, audio_np = audio
        else:
            audio_np = audio
            sr = 16000
        if len(audio_np.shape) > 1:
            audio_np = np.mean(audio_np, axis=1)
        if not np.issubdtype(audio_np.dtype, np.floating):
            if audio_np.dtype == np.int16:
                audio_np = audio_np.astype(np.float32) / 32768.0
            else:
                audio_np = audio_np.astype(np.float32)
        if sr != 16000:
            import librosa
            audio_np = librosa.resample(audio_np, orig_sr=sr, target_sr=16000)
            sr = 16000
        inputs = processor(audio_np, sampling_rate=sr, return_tensors="pt")
        input_features = inputs.input_features.to(device)
        with torch.no_grad():
            predicted_ids = model.generate(input_features, task="transcribe", language=lang)
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        if reference_text and reference_text.strip():
            wer = 100 * metric.compute(predictions=[transcription], references=[reference_text.strip()])
            return transcription, f"{wer:.2f}%"
        else:
            return transcription, "-"
    demo = gr.Interface(
        fn=transcribe_and_score,
        inputs=[
            gr.Audio(sources=["upload", "microphone"], type="numpy", label="Audio (upload or record)"),
            gr.Textbox(lines=2, label="Reference Text (optional)", placeholder="Paste ground truth here if you want WER computed")
        ],
        outputs=[
            gr.Textbox(label="Transcription"),
            gr.Textbox(label="WER")
        ],
        title="Whisper Fine-tuned in Assamese",
        description="Upload or record an audio file."
    )
    demo.launch()